# 📜 Historical Digital Edition Workflow

Interactive walkthrough of the two-stage OCR → NER pipeline for historical German books.

**Stages:**
1. Download images via IIIF
2. OCR — transcribe Fraktur pages into structured JSON
3. NER — annotate named entities
4. Export — interactive HTML edition + CSV

> **Colab users:** Set  in *Secrets* (🔑 icon, left sidebar) before running.

## 0. Install dependencies

In [ ]:
!pip install -q google-genai pillow tqdm python-dotenv

## 1. Environment setup

In [ ]:
import os, sys

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    api_key = userdata.get("GEMINI_API_KEY")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Set GEMINI_API_KEY in Colab Secrets or .env")
print("✅ API key ready")

## 2. Paths & config

In [ ]:
if IN_COLAB:
    IMAGE_FOLDER  = "/content/drive/MyDrive/digital_edition/images"
    OUTPUT_FOLDER = "/content/drive/MyDrive/digital_edition/output"
    REPO_ROOT = "/content/historical-digital-edition"
else:
    IMAGE_FOLDER  = "../images"
    OUTPUT_FOLDER = "../output"
    REPO_ROOT = os.path.abspath("..")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src import config
print(f"Images : {IMAGE_FOLDER}")
print(f"Output : {OUTPUT_FOLDER}")
print(f"Model  : {config.MODEL_ID}")

## 3. Download images (IIIF)

In [ ]:
from src.downloader import IIIFDownloader

downloader = IIIFDownloader(
    book_id=config.BOOK_ID,
    output_dir=IMAGE_FOLDER,
    delay=config.DOWNLOAD_DELAY_SECONDS,
)
downloaded = downloader.download(
    start_seq=config.DOWNLOAD_START_SEQ,
    end_seq=config.DOWNLOAD_END_SEQ,
)
print(f"Downloaded {len(downloaded)} images")

## 4. Test single page

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from google import genai
from src import process_page
from src.pipeline import get_image_files

client = genai.Client(api_key=api_key)
image_files = get_image_files(IMAGE_FOLDER)
print(f"Found {len(image_files)} images")

test_image = image_files[2]  # change index as needed
test_result = process_page(
    client=client,
    image_path=test_image,
    page_number=1,
    entity_types=config.ENTITY_TYPES,
    model_id=config.MODEL_ID,
    thinking_level=config.THINKING_LEVEL,
)

print("
--- OCR (first 500 chars) ---")
print(test_result.ocr_text[:500])
print(f"
--- Entities ({len(test_result.entities)}) ---")
for e in test_result.entities[:15]:
    print(f"  [{e.entity_type}] "{e.text}"")

## 5. Process full book

In [ ]:
from src import process_book

results = process_book(
    client=client,
    image_folder=IMAGE_FOLDER,
    output_folder=OUTPUT_FOLDER,
    entity_types=config.ENTITY_TYPES,
    model_id=config.MODEL_ID,
    thinking_level=config.THINKING_LEVEL,
    start_page=0,
    end_page=None,  # set e.g. to 10 for a quick test
)

total_ents = sum(len(r.entities) for r in results)
print(f"Pages: {len(results)} | Entities: {total_ents}")

## 6. Generate HTML edition

In [ ]:
from pathlib import Path
from src import generate_html_edition

html_path = generate_html_edition(
    results=results,
    output_path=Path(OUTPUT_FOLDER) / "digital_edition.html",
    title="Historische Digitalausgabe",
    entity_colors=config.ENTITY_COLORS,
    entity_labels=config.ENTITY_LABELS,
    image_folder=IMAGE_FOLDER,  # None to skip facsimile embedding
)
print(f"HTML edition: {html_path}")

if IN_COLAB:
    from google.colab import files
    files.download(str(html_path))

## 7. Export entities to CSV

In [ ]:
import csv
from pathlib import Path

csv_path = Path(OUTPUT_FOLDER) / "all_entities.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["page","entity_type","text","start_char","end_char","context"])
    for r in results:
        for e in r.entities:
            writer.writerow([r.page_number,e.entity_type,e.text,e.start_char,e.end_char,e.context])

print(f"CSV saved: {csv_path}")

freq = {}
for r in results:
    for e in r.entities:
        freq[(e.entity_type, e.text)] = freq.get((e.entity_type, e.text), 0) + 1

print("
Top 20 entities:")
for (etype, text), count in sorted(freq.items(), key=lambda x: -x[1])[:20]:
    print(f"  [{etype}] "{text}": {count}x")